# wandb-log-step — worked example 3: Log validation metrics at the end of each epoch

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `wandb-log-step`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Validation metrics are typically logged less frequently than training loss — once per epoch rather than once per batch. We can share the same `examples_seen` x-axis by passing the same `step=examples_seen` value at the end of each epoch, aligning validation metrics with the last training step of that epoch on the wandb chart.

## Worked solution

**Step 1 — outer loop over epochs, inner loop over batches.**
We structure the loop with an outer epoch loop and an inner batch loop. The `examples_seen` counter still increments per batch, maintaining a global count across all epochs.

**Step 2 — log training loss per batch.**
Inside the inner loop, we log `{'train/loss': loss}` at each batch's `examples_seen` value. This produces one log point per batch.

**Step 3 — log validation loss per epoch.**
After the inner loop (at the end of each epoch), we log `{'val/loss': val_loss}` at the SAME `step=examples_seen` as the last batch of that epoch. Both metrics share the same x-axis, so you can overlay them on the wandb dashboard.

In [ ]:
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb

def log_train_and_val_epochs(train_data_per_epoch, val_losses, batch_size):
    """
    train_data_per_epoch: list of lists; outer=epochs, inner=batch losses
    val_losses: list of floats, one per epoch
    batch_size: int
    Returns: final examples_seen
    """
    examples_seen = 0
    for epoch_idx, batch_losses in enumerate(train_data_per_epoch):
        # Per-batch logging
        for loss in batch_losses:
            examples_seen += batch_size
            wandb.log({'train/loss': loss}, step=examples_seen)
        # Per-epoch validation logging (same step as last batch)
        wandb.log({'val/loss': val_losses[epoch_idx]}, step=examples_seen)
    return examples_seen

# Exercise it
wandb.log.reset_mock()
train_data = [
    [2.5, 2.1, 1.8],  # epoch 0: 3 batches
    [1.5, 1.2, 1.0],  # epoch 1: 3 batches
]
val_losses = [1.9, 1.1]
final = log_train_and_val_epochs(train_data, val_losses, batch_size=32)
print('Final examples_seen:', final)  # 192
print('Total log calls:', wandb.log.call_count)  # 8: 6 train + 2 val